# 01 · Quickstart

The whole workflow in one notebook: **raw events → sequences → train → predict → save**.
The following cookbooks go deeper into each step:

| Notebook | Topic |
|---|---|
| 01_quickstart | the whole workflow in one notebook |
| 02_data_preparation | how samples must look, splitting, vocabulary, scaling, `CustomCollate` |
| 03_custom_collate_functions | targets: next value, delta, ratio, buckets, your own rule |
| 04_features_and_model | the task types, model configuration and validation |
| 05_training | `fit`, history, best epoch, learned task weights |
| 06_prediction_and_saving | evaluation vs. real prediction, decoding, `save` / `load` |
| 07_prediction_intervals | positive values, intervals with any coverage, calibration |
| 08_custom_training_loop | the lower-level building blocks for your own loop |

The notebooks use `pandas` for display: `pip install next-sequence pandas`.

In [1]:
import next_sequence as ns
print("next-sequence", ns.__version__, "| device:", ns.default_device())

next-sequence 0.1.0 | device: cuda


## The example data

Every cookbook uses the same small synthetic shop: each row is one purchase with an `item`,
a `price` and a `weekday`. Shoppers buy the five items in a fixed cycle
(`apple → bread → cheese → dates → eggs → apple ...`), starting at a random item, so the next
item is learnable and we can check the model gets it right.

In [2]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

events = make_events()
events.head(8)

,user_id,t,item,price,weekday
0,0,0,dates,102.41,0
1,0,1,eggs,57.11,4
2,0,2,apple,19.94,6
3,0,3,bread,47.97,3
4,0,4,cheese,174.89,1
5,0,5,dates,95.09,2
6,0,6,eggs,55.68,0
7,0,7,apple,20.47,2


## 1. One sequence per user

The model reads **one sample per sequence**: a dict where every feature is a list ordered by time.

In [3]:
users = (events.sort_values(["user_id", "t"])
               .groupby("user_id").agg(list).reset_index()
               .drop(columns="t").to_dict(orient="records"))
{k: (v[:5] + ["..."] if isinstance(v, list) else v) for k, v in users[0].items()}   # first 5 steps

{'user_id': 0,
 'item': ['dates', 'eggs', 'apple', 'bread', 'cheese', '...'],
 'price': [102.41, 57.11, 19.94, 47.97, 174.89, '...'],
 'weekday': [0, 4, 6, 3, 1, '...']}

## 2. Split by time

Leave-last-out: for each user, **train** predicts the 3rd-to-last event, **val** the 2nd-to-last
and **test** the last one. The last element of every list is the target.

In [4]:
def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]
val_raw = [cut(u, -1) for u in users]
test_raw = [cut(u, None) for u in users]
print("user 0 items | train:", train_raw[0]["item"][-3:], "| test:", test_raw[0]["item"][-3:])

user 0 items | train: ['apple', 'bread', 'cheese'] | test: ['cheese', 'dates', 'eggs']


## 3. Preprocessing, fitted on the training split only

A vocabulary for `item` (`PAD=0`, `UNK=1`), and a scaler for `price` (standardized `log1p`).

In [5]:
import math
from next_sequence import create_bow

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]}          # 0 stays free for padding
            for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)
print(bow)

{'PAD': 0, 'UNK': 1, 'apple': 2, 'bread': 3, 'cheese': 4, 'dates': 5, 'eggs': 6}


## 4. Features, model configuration, collate functions and DataLoaders

- `item`: categorical → predict the next item (classification)
- `price`: numeric → predict the next price (regression)
- `weekday`: categorical → input only (passthrough)

`make_collate` builds the per-feature collate functions: in training mode the last step is the
target. `CustomCollate.from_config` takes `max_len` and the input order from the model
configuration, so the data always matches what the model expects.

In [6]:
from torch.utils.data import DataLoader
from next_sequence import CoreDataset, CustomCollate, Feature, NextSequenceConfig, TaskType, make_collate

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
config = NextSequenceConfig(features=features, max_len=16, emb_dim=32, attn_heads=2)
funcs = {"item": make_collate("long"), "price": make_collate("float"), "weekday": make_collate("long")}

def loader(samples, inference=False, shuffle=False):
    collate = CustomCollate.from_config(config, "item", funcs, inference=inference)
    return DataLoader(CoreDataset(samples), batch_size=32, shuffle=shuffle, collate_fn=collate)

train_loader, val_loader, test_loader = loader(train, shuffle=True), loader(val), loader(test)

## 5. Train

In [7]:
from next_sequence import NextSequence

model = NextSequence(config, epochs=15, lr=3e-3, seed=0)
model.fit(train_loader, val_loader)
print("best epoch:", model.best_epoch_)

epoch=1 | train_loss=2.4540 | train_item=1.7857 | train_price=1.3515 | val_loss=1.9034 | val_item=1.5804 | val_price=0.6857
epoch=2 | train_loss=1.5506 | train_item=1.3007 | train_price=0.5411 | val_loss=1.2984 | val_item=1.0942 | val_price=0.4572


epoch=3 | train_loss=0.9820 | train_item=0.8183 | train_price=0.3599 | val_loss=0.6405 | val_item=0.5516 | val_price=0.1947
epoch=4 | train_loss=0.4758 | train_item=0.3964 | train_price=0.1654 | val_loss=0.3478 | val_item=0.2663 | val_price=0.1653


epoch=5 | train_loss=0.2040 | train_item=0.1676 | train_price=0.0778 | val_loss=0.1084 | val_item=0.0889 | val_price=0.0534
epoch=6 | train_loss=0.0696 | train_item=0.0611 | train_price=0.0434 | val_loss=0.0273 | val_item=0.0360 | val_price=0.0283


epoch=7 | train_loss=0.0181 | train_item=0.0320 | train_price=0.0318 | val_loss=-0.0068 | val_item=0.0231 | val_price=0.0203
epoch=8 | train_loss=-0.0116 | train_item=0.0206 | train_price=0.0276 | val_loss=-0.0305 | val_item=0.0148 | val_price=0.0194


epoch=9 | train_loss=-0.0326 | train_item=0.0152 | train_price=0.0244 | val_loss=-0.0463 | val_item=0.0122 | val_price=0.0178
epoch=10 | train_loss=-0.0478 | train_item=0.0130 | train_price=0.0211 | val_loss=-0.0582 | val_item=0.0104 | val_price=0.0165


epoch=11 | train_loss=-0.0594 | train_item=0.0116 | train_price=0.0176 | val_loss=-0.0665 | val_item=0.0095 | val_price=0.0154
epoch=12 | train_loss=-0.0651 | train_item=0.0111 | train_price=0.0183 | val_loss=-0.0714 | val_item=0.0092 | val_price=0.0147


epoch=13 | train_loss=-0.0681 | train_item=0.0107 | train_price=0.0196 | val_loss=-0.0745 | val_item=0.0089 | val_price=0.0137
epoch=14 | train_loss=-0.0724 | train_item=0.0103 | train_price=0.0156 | val_loss=-0.0758 | val_item=0.0088 | val_price=0.0131


epoch=15 | train_loss=-0.0716 | train_item=0.0101 | train_price=0.0180 | val_loss=-0.0760 | val_item=0.0088 | val_price=0.0131
best epoch: 15


## 6. Predict and decode

`predict` returns raw outputs: logits for `item`, scaled values for `price`. Decoding is yours.

In [8]:
import torch

out = model.predict(test_loader, mask_pad=True)
print({name: tuple(t.shape) for name, t in out.items()})

id2item = {i: t for t, i in bow.items()}
logits = out["item"].clone()
logits[:, bow["UNK"]] = float("-inf")                    # never predict "unknown"
pred_item = [id2item[i] for i in logits.argmax(-1).tolist()]
pred_price = torch.expm1(out["price"].squeeze(-1) * price_std + price_mean)

result = pd.DataFrame({
    "true_item": [u["item"][-1] for u in test_raw], "pred_item": pred_item,
    "true_price": [u["price"][-1] for u in test_raw], "pred_price": pred_price.double().numpy().round(2),
})
print(f"item accuracy: {(result.true_item == result.pred_item).mean():.3f} | "
      f"price MAE: {(result.true_price - result.pred_price).abs().mean():.2f}")
result.head()

{'item': (300, 7), 'price': (300, 1)}
item accuracy: 1.000 | price MAE: 5.16


,true_item,pred_item,true_price,pred_price
0,eggs,eggs,57.72,56.05
1,bread,bread,40.51,40.38
2,dates,dates,102.13,95.44
3,cheese,cheese,173.41,185.56
4,dates,dates,87.02,95.68


## 7. Predict the *next* event

`inference=True` feeds the **full** history and predicts what comes after it.

In [9]:
future = model.predict(loader(test, inference=True), mask_pad=True)["item"]
future[:, bow["UNK"]] = float("-inf")
pd.DataFrame({"last_item": [u["item"][-1] for u in test_raw[:5]],
              "predicted_next": [id2item[i] for i in future.argmax(-1)[:5].tolist()]})

,last_item,predicted_next
0,eggs,apple
1,bread,cheese
2,dates,eggs
3,cheese,dates
4,dates,eggs


## 8. Save and load, together with your preprocessing

In [10]:
model.save("outputs/quickstart.pt", metadata={"bow": bow, "price_mean": price_mean, "price_std": price_std})

loaded = NextSequence.load("outputs/quickstart.pt")
same = torch.allclose(loaded.predict(test_loader)["price"], model.predict(test_loader)["price"], atol=1e-5)
print(loaded, "| same predictions:", same, "| metadata keys:", list(loaded.metadata_))

NextSequence(features=['item', 'price', 'weekday'], epochs=15, lr=0.003, device='cuda', fitted) | same predictions: True | metadata keys: ['bow', 'price_mean', 'price_std']
